Import des bibliothèques et déclaration des variables globales

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import random as rd
from collections import Counter
from threading import Thread
from math import *
from time import *

gene = 5
individu = 5*gene
critere_arret = 0
Pc = 0.8
Pm = 1/gene
population = []
liste_fitness = []
chemin = []
parent = []
newPopulation = []
x = []
y = []

Code

In [2]:
def compareList(l1, l2):
    if not (isinstance(l1, list) and isinstance(l2, list)): return False
    if not all (isinstance(sous, list) for sous in l1+l2): return False

    t1 = [tuple(sous) for sous in l1]
    t2 = [tuple(sous) for sous in l2]
    
    return Counter(t1) == Counter(t2)

Classe pour voir graphiquement le trajet pris par le voyageur

In [3]:
class Voyageur():
    def __init__(self, trajets): #trajets = liste des trajets
        self.trajets = trajets
        
    def realiser(self):
        print('départ du voyageur')
        self.affreter_train()
        self.voyage()
        
    def affreter_train(self):
        self.train = Train(self.trajets[0].provenance.xville, self.trajets[0].provenance.yville)
        
    def voyage(self): #permet de faire plusieurs trajets
        global gene
        for i in range (gene-1):
            self.train.voyage(self.trajets[i].destination.xville, self.trajets[i].destination.yville)

Classe de création du train

In [4]:
class Train():
    def __init__(self, xt, yt):
        self.xt = xt
        self.yt = yt
        self.rayon = 30

        colors = 'purple'
        rayon = self.rayon
        points = [xt-rayon, yt-rayon/2, xt+rayon, yt-rayon/2, xt+rayon, yt+rayon/2, xt-rayon, yt+rayon/2]
        self.wagon = can.create_polygon(points, fill = 'purple', width = 2)
            
    def voyage(self, xv, yv): 
        print('passage d\'une ville à une autre...')
        steps = 50  # nombre d'étapes pour l'animation
        x_start, y_start = self.xt, self.yt
        dx = (xv - x_start) / steps
        dy = (yv - y_start) / steps

        for _ in range(steps):
            can.move(self.wagon, dx, dy)
            self.xt += dx
            self.yt += dy
            fen.update()  # rafraîchit le canvas
            sleep(0.02)   # vitesse de l'animation
            
    def rafraichissement_position(self, i): #rafraichit l'affichage du train
        rayon = self.rayon
        p = [self.xt-rayon, self.yt-rayon/2, self.xt+rayon, self.yt-rayon/2, self.xt+rayon, self.yt+rayon/2, self.xt-rayon, self.yt+rayon/2]
        can.coords(self.wagon, p) 
        can.update()

Classe de création des villes (graphiquement)

In [5]:
class Ville(): #pas de modification
    def __init__(self, nom, position_x, position_y):
        self.nom = nom
        self.xville = position_x
        self.yville = position_y
        can.create_oval(self.xville-5, self.yville-5, self.xville+5, self.yville+5, fill='black')  
        can.create_text(self.xville, self.yville-15, text=self.nom, fill="black",font=('Arial', 14))

Classe donnant le chemin à emprunter pour le voyageur

In [6]:
class Chemin():
    def __init__(self, villes):
        self.villes = villes

Classe de création de tous les chemins entre les villes

In [7]:
class Trajet():
    def __init__(self, provenance, destination):
        self.provenance = provenance
        self.destination = destination
        can.create_line(self.provenance.xville, self.provenance.yville, self.destination.xville, self.destination.yville, fill = 'red', width = 2)

Fonction d'évaluation des individus

In [8]:
def fitness():
    global chemin, x, y
    distance = 0.0
    xy=np.column_stack((x[chemin],y[chemin]))
    distance = np.sum(np.sqrt(np.sum((xy-np.roll(xy,-1,axis=0))**2,axis=1)))
    return distance

Fonction de génération de la première génération d'individus aléatoirement

In [9]:
def firstGeneration(i):
    global population, chemin, gene
    for j in range(0,gene,1):
        bool = 1
        if j==0 : element= rd.randint(0,gene-1)
        else : 
            while bool ==1:
                element = rd.randint(0,gene-1)
                l=0
                for k in range(0,j,1):
                    if population [i][k] == element :
                        bool = 1
                        l = l+1
                if l== 0: bool=0
                    
        population[i][j] = element
        chemin[j] = population[i][j] 

Fonction qui calcule pour chaque individu sa fitness

In [10]:
def evaluation(init):
    global population, liste_fitness, chemin, individu, gene
    liste_fitness = []
    for i in range (0,individu,1):
        if(init): 
            population.append([0]*gene)
            firstGeneration(i)
        chemin = population[i]
        d = fitness()
        liste_fitness.append(d)

Fonction sélectionne les parents

In [11]:
def selectionParents():
    global population, liste_fitness, M
    z = zip(population,liste_fitness)
    s = sorted(z, key=lambda element: element[1])
    parent = s[:individu//2]
    return parent

Fonction qui garde aléatoirement 2 parents parmis ceux sélectionnés

In [12]:
def selection():
    global parent, gene, individu
    select = []
    parent1 = rd.randint(0, individu//2-1)
    select.append(parent[parent1][0])
    same = True
    while same:
        parent2 = rd.randint(0, individu//2-1)
        if parent1 != parent2: same = False
    select.append(parent[parent2][0])
    return select

Fonction pour supprimer les doublons dans les enfants croisés

In [13]:
def oxCrossover(parent1, parent2, a, b):
    global gene
    enfant = [0]*gene
    
    enfant[a:b] = parent1[a:b]
    partParent1 = set(parent1[a:b])
    
    posEnf = b%gene
    posParent2 = b%gene
    done = gene - (b - a)
    count = 0
    
    while count < done:
        ville = parent2[posParent2%gene]
        if ville not in partParent1:
            enfant[posEnf%gene] = ville
            posEnf = (posEnf+1)%gene
            count += 1
        posParent2 = (posParent2 + 1)%gene
    return enfant

Fonction qui croise les parents pour donner des enfants

In [14]:
def crossover(parent1, parent2):
    global individu, gene
    a = rd.randint(0, gene-1)
    b = rd.randint(0, gene-1)
    enfant1 = oxCrossover(parent1, parent2, a, b)
    enfant2 = oxCrossover(parent1, parent2, a ,b)
    return [enfant1, enfant2]

Fonction qui fait la mutation des enfants

In [15]:
def swapMutation(enfant):
    global M
    positionMutation1 = rd.randint(0,gene-1)
    positionMutation2 = rd.randint(0,gene-1)
    temp = enfant[positionMutation1]
    enfant[positionMutation1] = enfant[positionMutation2]
    enfant[positionMutation2] = temp
    return enfant

Fonction qui garde l'enfant ou le parent le plus pertinent

In [16]:
def mimetique(enfantsCandidats):
    global chemin
    enfantsFitness = []

    for enfant in enfantsCandidats:
        chemin = enfant
        enfantsFitness.append(fitness())
    
    z = zip(enfantsCandidats, enfantsFitness)
    s = sorted(z, key = lambda element : element[1])
    return s[0][0]

Fonction qui renvoie le meilleur enfant ou parent

In [17]:
def creationEnfant(Pc, Pm):
    global parent
    enfantsCandidats = []
    
    select = selection()
    parent1 = select[0]
    parent2 = select[1]

    crossoverProba = rd.uniform(0.0,1.0)
    if crossoverProba <= Pc: select = crossover(parent1, parent2)
    enfant1 = select[0]
    enfant2 = select[1]

    mutationProba = rd.uniform(0.0, 1.0)
    if mutationProba < Pm: 
        enfant1 = swapMutation(enfant1)
        enfant2 = swapMutation(enfant2)

    enfantsCandidats = [parent1, parent2, enfant1, enfant2]

    return mimetique(enfantsCandidats)

Fonction principale de l'algo

In [18]:
def algo():
    global population, liste_fitness, parent, newPopulation, x, y, chemin, gene, individu, critere_arret, Pc, Pm
    print(gene)
    population = []
    liste_fitness = []
    parent = []
    newPopulation = []
    oldBest = None
    newBest = []

    #cheminV (liste d'objets ville)
    #trajets (liste d'objets trajet)
        
    x=np.random.randint(0,1500,gene)
    y=np.random.randint(0,700,gene)
    chemin = np.arange(gene) 
    
    init = True

    print('entrée dans l\'algo')
    while critere_arret <= int(4*gene):
        evaluation(init)
        if init: init = False
        parent = selectionParents()
        newBest = parent[0][1]
        for i in range (0,individu,1):
            newPopulation.append(creationEnfant(Pc, Pm))
        if oldBest is not None and abs(newBest - oldBest) == 0: 
            critere_arret += 1
        else: critere_arret = 0
        oldBest = newBest
        population = newPopulation[:]
        newPopulation = []

    parent = selectionParents()
    chemin = parent[0][0]
    print('Meilleur chemin obtenu par l\'algo : ', chemin, ', sa fitness est de : ', parent[0][1])
    
    villes = []
    trajets = []
    for i in range(gene):
        villes.append(Ville(f"V{i}", x[i], y[i]))
    for i in range(gene-1):
        print('trajet load')
        trajets.append(Trajet(villes[chemin[i]], villes[chemin[i+1]]))

    voyageur = Voyageur(trajets)
    voyageur.realiser()

    #boucle for pour créer toutes les villes à afficher
    #boucle for pour créer tous les trajets (bouts du chemin)
    #creation de Voyageur
    #appel à sa fonction realiser

Création du thread pour exécuter le code

In [19]:
def new_voyageur():
    i = 0
    thread_voyageur = Thread(target = algo)
    thread_voyageur.start()

Création de la fenêtre d'affichage du trajet du voyageur

In [20]:
from tkinter import *

fen = Tk()

can_width = 1500
can_height = 700
can = Canvas(fen, width = can_width, height = can_height, bg = 'white')
can.pack(side = TOP, padx = 5, pady = 5)

button_quit = Button(fen, text = 'Quitter', command = fen.destroy)
button_quit.pack(side = BOTTOM)

new_voyageur()

fen.mainloop()

5
entrée dans l'algo
Meilleur chemin obtenu par l'algo :  [2, 1, 4, 3, 0] , sa fitness est de :  2117.91215331918
trajet load
trajet load
trajet load
trajet load
départ du voyageur
passage d'une ville à une autre...
passage d'une ville à une autre...
passage d'une ville à une autre...
passage d'une ville à une autre...
